In [31]:
import os
import gdown
import zipfile

file_id = "1dAHaPjCjqaOnsZVZtLNYXXvfK6eHnfQS"
url = f"https://drive.google.com/uc?id={file_id}"

downloaded_file = gdown.download(url, output="dataset.zip", quiet=False, fuzzy=True)

print("size:", os.path.getsize(downloaded_file))
with open(downloaded_file, "rb") as f:
    print("magic:", f.read(4))

extract_folder = '.'
with zipfile.ZipFile(downloaded_file, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

Downloading...
From (original): https://drive.google.com/uc?id=1dAHaPjCjqaOnsZVZtLNYXXvfK6eHnfQS
From (redirected): https://drive.google.com/uc?id=1dAHaPjCjqaOnsZVZtLNYXXvfK6eHnfQS&confirm=t&uuid=466ff377-1ad4-48a2-a64f-a2a982501280
To: /content/dataset.zip
100%|██████████| 109M/109M [00:00<00:00, 173MB/s]


size: 108744812
magic: b'PK\x03\x04'


In [ ]:
import os, re
from pathlib import Path
from collections import deque, Counter
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

_paddle = None
_easy = None
try:
    from paddleocr import PaddleOCR
    for kw in (dict(lang='en', use_angle_cls=False, show_log=False), dict(lang='en'), {}):
        try:
            _paddle = PaddleOCR(**kw); break
        except Exception: continue
except Exception: pass

try:
    import easyocr, torch
    _easy = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
except Exception: pass

TEST_DIR = Path("mini-packman-dataset/test/images")
OUT_CSV  = Path("submission.csv")
ROWS, COLS = 15, 21
CELL = 40
W, H = COLS * CELL, ROWS * CELL
V = ROWS * COLS

def order_quad(pts):
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(1); d = np.diff(pts, axis=1).ravel()
    return np.array([pts[np.argmin(s)], pts[np.argmin(d)],
                     pts[np.argmax(s)], pts[np.argmax(d)]], dtype=np.float32)

def blue_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, (90, 100, 100), (135, 255, 255))

def find_maze_quad(img):
    bm = blue_mask(img)
    bm = cv2.morphologyEx(bm, cv2.MORPH_CLOSE, np.ones((11,11), np.uint8))
    cnts, _ = cv2.findContours(bm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    big = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(big) < 0.05 * img.shape[0] * img.shape[1]:
        return None
    hull = cv2.convexHull(big)
    peri = cv2.arcLength(hull, True)
    for eps in (0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2):
        ap = cv2.approxPolyDP(hull, eps*peri, True)
        if len(ap) == 4:
            return order_quad(ap.reshape(4,2).astype(np.float32))
    return order_quad(cv2.boxPoints(cv2.minAreaRect(hull)).astype(np.float32))

def warp_perspective(img, quad):
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(quad, dst)
    return cv2.warpPerspective(img, M, (W, H), flags=cv2.INTER_CUBIC)

def warp_bbox(img):
    bm = blue_mask(img)
    bm = cv2.morphologyEx(bm, cv2.MORPH_CLOSE, np.ones((9,9), np.uint8))
    cnts, _ = cv2.findContours(bm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    big = max(cnts, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(big)
    if w < 50 or h < 50: return None
    return cv2.resize(img[y:y+h, x:x+w], (W, H), interpolation=cv2.INTER_CUBIC)

def build_wall_confidence(warped):
    hsv = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
    Hc, Sc, Vc = hsv[...,0], hsv[...,1], hsv[...,2]
    blue = ((Hc > 90) & (Hc < 135) & (Sc > 100) & (Vc > 100)).astype(np.float32)
    blue = cv2.GaussianBlur(blue, (5,5), 1.5)
    cR = np.zeros((ROWS, COLS), np.float32)
    cD = np.zeros((ROWS, COLS), np.float32)
    pad = 8
    for r in range(ROWS):
        for c in range(COLS):
            if c + 1 < COLS:
                x = (c+1)*CELL
                s = blue[r*CELL+pad:(r+1)*CELL-pad, max(0,x-3):min(W,x+4)]
                if s.size: cR[r,c] = float(s.max(axis=1).mean())
            if r + 1 < ROWS:
                y = (r+1)*CELL
                s = blue[max(0,y-3):min(H,y+4), c*CELL+pad:(c+1)*CELL-pad]
                if s.size: cD[r,c] = float(s.max(axis=0).mean())
    return cR, cD

def build_tree(cR, cD):
    edges = []
    for r in range(ROWS):
        for c in range(COLS):
            i = r*COLS + c
            if c+1 < COLS: edges.append((1.0 - cR[r,c], i, i+1))
            if r+1 < ROWS: edges.append((1.0 - cD[r,c], i, i+COLS))
    edges.sort(reverse=True)
    par = list(range(V))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    g = [[] for _ in range(V)]
    used = 0
    for _, i, j in edges:
        ri, rj = find(i), find(j)
        if ri != rj:
            par[ri] = rj
            g[i].append(j); g[j].append(i)
            used += 1
            if used == V - 1: break
    return g

def tree_clarity(cR, cD):
    v = np.concatenate([cR.flatten(), cD.flatten()])
    return float(((v < 0.15) | (v > 0.7)).mean())

def yellow_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return cv2.inRange(hsv, (15, 100, 120), (40, 255, 255))

def red_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return cv2.bitwise_or(
        cv2.inRange(hsv, (0, 120, 80), (10, 255, 255)),
        cv2.inRange(hsv, (170, 120, 80), (180, 255, 255)))

def cell_count(mask, r, c, pad=4):
    y0, y1 = r*CELL+pad, (r+1)*CELL-pad
    x0, x1 = c*CELL+pad, (c+1)*CELL-pad
    return int(mask[y0:y1, x0:x1].sum() // 255)

def detect_objects(warped):
    ym = yellow_mask(warped)
    cy = np.zeros((ROWS, COLS), np.int32)
    for r in range(ROWS):
        for c in range(COLS):
            cy[r,c] = cell_count(ym, r, c)
    pac_r, pac_c = np.unravel_index(cy.argmax(), cy.shape)
    pac = (int(pac_r), int(pac_c))
    if cy[pac_r, pac_c] < 40:          # پک‌من دیده نشد
        return None, set()
    pellets = {(r,c) for r in range(ROWS) for c in range(COLS)
               if (r,c) != pac and cy[r,c] >= 8}
    return pac, pellets

def detect_cherry(warped):
    rm = red_mask(warped)
    cr = np.zeros((ROWS, COLS), np.int32)
    for r in range(ROWS):
        for c in range(COLS):
            cr[r,c] = cell_count(rm, r, c)
    r, c = np.unravel_index(cr.argmax(), cr.shape)
    if cr[r,c] < 15: return None
    return (int(r), int(c))

def bfs_path(g, start, goal):
    if start is None or goal is None: return None
    s, t = start[0]*COLS+start[1], goal[0]*COLS+goal[1]
    q = deque([s]); par = {s: None}
    while q:
        u = q.popleft()
        if u == t: break
        for v in g[u]:
            if v not in par:
                par[v] = u; q.append(v)
    if t not in par: return None
    seq, cur = [], t
    while cur is not None:
        seq.append((cur // COLS, cur % COLS)); cur = par[cur]
    return seq[::-1]

def cells_to_str(cells):
    out = []
    for (r1,c1),(r2,c2) in zip(cells, cells[1:]):
        dr, dc = r2-r1, c2-c1
        if dr==-1 and dc==0: out.append('U')
        elif dr==1 and dc==0: out.append('D')
        elif dr==0 and dc==-1: out.append('L')
        elif dr==0 and dc==1: out.append('R')
        else: return ""
    return "".join(out)

def _parse_paddle(res):
    if not res: return ""
    page = res[0] if (isinstance(res, list) and res and isinstance(res[0], list)) else res
    if page is None: return ""
    if not isinstance(page, list): page = [page]
    texts = []
    for it in page:
        if isinstance(it, (list, tuple)) and len(it) == 2:
            if isinstance(it[1], (list, tuple)) and it[1]:
                texts.append(str(it[1][0]))
        elif isinstance(it, dict):
            for k in ('rec_texts','texts','text'):
                if k in it:
                    v = it[k]
                    if isinstance(v, list): texts.extend(str(x) for x in v)
                    else: texts.append(str(v))
    return re.sub(r"\D", "", "".join(texts))

def paddle_digits(bgr):
    if _paddle is None: return ""
    for kw in (dict(cls=False), dict(use_textline_orientation=False), {}):
        try: return _parse_paddle(_paddle.ocr(bgr, **kw))
        except Exception: continue
    return ""

def easy_digits(bgr):
    if _easy is None: return ""
    try:
        res = _easy.readtext(bgr, allowlist="0123456789", detail=0, paragraph=False)
        return re.sub(r"\D", "", "".join(str(x) for x in res)) if res else ""
    except Exception: return ""

def digits_to_score(digits):
    if not digits: return 0
    s = digits[-5:] if len(digits) >= 5 else digits.rjust(5, '0')
    try: v = int(s)
    except Exception: return 0
    if v == 0: return 0
    if v % 10 == 0: return v

    try:
        v2 = int(s[:4] + '0')
        return v2 if v2 > 0 else 0
    except Exception: return 0

def read_initial_score(img):
    h, w = img.shape[:2]
    cands = []
    for x0r, x1r, y0r, y1r in [(0.05, 0.30, 0.72, 0.82), (0.06, 0.29, 0.73, 0.81)]:
        roi = img[int(h*y0r):int(h*y1r), int(w*x0r):int(w*x1r)]
        if roi.size == 0: continue
        big = cv2.resize(roi, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
        for fn in (easy_digits, paddle_digits):
            s = digits_to_score(fn(big))
            if s > 0: cands.append(s)
            if len(cands) >= 2 and len(set(cands)) == 1:
                return cands[0]
        gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        thc = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)
        for fn in (easy_digits, paddle_digits):
            s = digits_to_score(fn(thc))
            if s > 0: cands.append(s)
    if not cands: return 0
    return Counter(cands).most_common(1)[0][0]

def process_case(path, debug=False):
    img = cv2.imread(str(path))
    name = Path(path).name
    base = {"case_filename": name, "initial_score": 0, "path": "",
            "yellow_pellets_on_path": 0, "final_score": 0}
    if img is None: return base

    initial = read_initial_score(img)

    best, best_clar = None, -1
    quad = find_maze_quad(img)
    if quad is not None:
        try:
            wp = warp_perspective(img, quad)
            cR, cD = build_wall_confidence(wp)
            cl = tree_clarity(cR, cD)
            if cl > best_clar: best, best_clar = (wp, cR, cD), cl
        except Exception: pass

    if best is None or best_clar < 0.5:
        try:
            wb = warp_bbox(img)
            if wb is not None:
                cR, cD = build_wall_confidence(wb)
                cl = tree_clarity(cR, cD)
                if cl > best_clar: best, best_clar = (wb, cR, cD), cl
        except Exception: pass

    if best is None:
        return {**base, "initial_score": initial, "final_score": initial}

    warped, cR, cD = best
    g = build_tree(cR, cD)
    pac, pellets = detect_objects(warped)
    cherry = detect_cherry(warped)

    if pac is None or cherry is None or pac == cherry:
        return {**base, "initial_score": initial, "final_score": initial}

    cells = bfs_path(g, cherry, pac)
    if not cells or len(cells) < 2:
        return {**base, "initial_score": initial, "final_score": initial}

    path_str = cells_to_str(cells)
    if not path_str:
        return {**base, "initial_score": initial, "final_score": initial}

    pset = set(cells)
    n_pel = sum(1 for c in pellets if c in pset and c != pac and c != cherry)
    final = initial + n_pel * 10 + 100

    if debug:
        cv2.imwrite(f"debug_{Path(path).stem}.png", warped)
        print(f"{name}: init={initial} pac={pac} cherry={cherry} "
              f"pellets={n_pel} pathlen={len(path_str)} final={final}")

    return {"case_filename": name, "initial_score": int(initial),
            "path": path_str, "yellow_pellets_on_path": int(n_pel),
            "final_score": int(final)}

paths = sorted(TEST_DIR.glob("*.png"))
print(f"Found {len(paths)} images")
rows = []
for p in tqdm(paths, desc="cases"):
    try: rows.append(process_case(p, debug=False))
    except Exception as e:
        print(f"ERR {p.name}: {e}")
        rows.append({"case_filename": p.name, "initial_score": 0, "path": "",
                     "yellow_pellets_on_path": 0, "final_score": 0})

df = pd.DataFrame(rows, columns=["case_filename","initial_score","path",
                                 "yellow_pellets_on_path","final_score"])
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV}")
print(f"initial>0: {(df['initial_score']>0).sum()}/{len(df)}")
print(f"mean path len: {df['path'].str.len().mean():.1f}")
print(f"mean pellets: {df['yellow_pellets_on_path'].mean():.1f}")
print(df.head())

/tmp/ipykernel_939/3288036544.py:16: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  _paddle = PaddleOCR(**kw); break
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/

Found 100 images


cases:   0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_939/3288036544.py:214: DeprecationWarning: Please use `predict` instead.
  try: return _parse_paddle(_paddle.ocr(bgr, **kw))


Wrote submission.csv
initial>0: 100/100
mean path len: 125.3
mean pellets: 55.2
   case_filename  initial_score  \
0  case_0000.png            770   
1  case_0001.png            620   
2  case_0002.png            600   
3  case_0003.png            570   
4  case_0004.png            960   

                                                path  yellow_pellets_on_path  \
0  DRRRURULURULULDDLLDLLULURRULURURDDRUUUUULUULLU...                      67   
1  LDDDDRRURUUUUULDLULULDLUURRUULDLULDDLDLUULDDDL...                      52   
2  ULLLLLDRDLDLDDDRURRRURDDLDDLLULLDRDLDRDDRDDRRR...                      50   
3  LDRDRUURDDDLLLDRDRRDLLLULUUUUUULUULDDDDRDDDDDR...                      47   
4  DDLULDDLUUUULDDDDDRRRURRURRULULULUUUUULDDDDDDL...                      86   

   final_score  
0         1540  
1         1240  
2         1200  
3         1140  
4         1920  


In [ ]:
print(f"FAILED         = {(df['path']=='').sum()}")                     # باید 0-1
print(f"pellets/10move = {(df['yellow_pellets_on_path']/df['path'].str.len().replace(0,1)*10).mean():.2f}")  # 3-5
print(f"mean pellets   = {df['yellow_pellets_on_path'].mean():.1f}")   # 35-45
print(f"mean path      = {df['path'].str.len().mean():.1f}")            # 85-95
density = df['yellow_pellets_on_path']/df['path'].str.len().replace(0,1)
print(f"density >65%   = {(density>0.65).sum()} cases")                 # باید 0-2

FAILED         = 6
pellets/10move = 4.07
mean pellets   = 35.1
mean path      = 82.1
density >65%   = 2 cases
